# Notebook 51 — Capstone III: Operate a Durable Multi-Agent System

    ## Learning objectives

    - Combine durable state, MCP tools, approvals, and specialist agents
- Evaluate trajectories, failure recovery, security, and capacity
- Produce an operational release with rollback and governance

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from the Colab Secrets UI without displaying it. Create a
# secret named exactly HF_TOKEN and enable notebook access with its toggle.
token = os.getenv("HF_TOKEN")
token_error = None
if IN_COLAB and not token:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        token_error = type(exc).__name__
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub environment variable.
if token:
    os.environ["HF_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("HF_TOKEN is unavailable. In Colab, open the key icon (Secrets), add HF_TOKEN, ")
    print("enable its Notebook access toggle, and rerun this cell. Public models still work.")
    if token_error:
        print("Colab secret lookup status:", token_error)


## 51.1 Workflow architecture

Choose a consequential but sandboxable task and first implement a deterministic workflow. Add one agent only for ambiguous decisions, then justify any specialist agents through context or permission isolation. Define typed state, events, task envelopes, tool schemas, terminal states, global budgets, and owners. Use MCP for composable capabilities without treating discovered servers as trusted. Draw every identity, data, and side-effect boundary before executing.


In [ ]:
architecture={"host":"policy + durable state","servers":["catalog","sandbox"],"agents":["supervisor","researcher","reviewer"],"human_gate":"external writes"}; print(architecture)


## 51.2 Durability and multi-agent coordination

Checkpoint before and after effects, issue stable idempotency keys, retain receipts, propagate cancellation, and resume after injected crashes. Human approval binds to an immutable proposed action and expires. A supervisor may route only registered roles; specialists receive least context and tools; parallel results merge deterministically with dissent preserved. Shared state is authoritative only through validated transitions, not agent-authored prose.


In [ ]:
states=["ready","planning","running","waiting_approval","committing","done","failed","cancelled"]; print(states)


## 51.3 Evaluation and adversarial testing

Create a deterministic environment and frozen tasks. Compare deterministic, single-agent, and multi-agent variants under equal model, permission, token, tool-call, and time budgets. Score task success, tool choice and arguments, effects, duplicated work, conflicts, intervention, steps, tokens, latency, and critical violations. Inject malformed calls, poisoned MCP resources, revoked permissions, duplicate delivery, worker loss, stale approval, sandbox attacks, and malicious specialists.


In [ ]:
faults=["crash_before_effect","crash_after_effect","malicious_resource","revoked_scope","worker_timeout","duplicate_message"]; print(faults)


## 51.4 Deployment and governance

Serve the pinned model through vLLM or the selected engine behind authenticated ingress and admission control. Trace state transitions and effects without retaining sensitive payloads unnecessarily. Canary with read-only scopes, rehearse rollback, and publish a system card containing versions, architecture, threat model, evaluations, known limitations, monitoring, ownership, and incident response. Conclude with an ablation-based answer to whether multiple agents improved the workflow enough to remain.


In [ ]:
comparison={"deterministic":.62,"single_agent":.81,"multi_agent":.84,"multi_agent_cost_ratio":2.3}; print(comparison,"keep only if bounded gains justify cost")


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## 51.5 Architecture ablation is the thesis

Implement deterministic workflow, single-agent, and multi-agent versions under matched model, permission, token, tool, and time budgets. Compare task success, effect correctness, duplicated work, conflicts, intervention, latency, and critical violations. The capstone should conclude whether role separation or parallelism produced enough measurable value to justify additional coordination and attack surface. Multiple agents are not the default success condition; a well-supported decision to remove them is equally valid.


In [ ]:
systems=[{"name":"workflow","success":.68,"cost":1.,"violations":0},{"name":"single","success":.82,"cost":2.,"violations":0},{"name":"multi","success":.84,"cost":4.5,"violations":0}];
for s in systems: print(s["name"],"success/cost",s["success"]/s["cost"])


## 51.6 Fault-injection and release drill

Automate crashes before and after effects, duplicate delivery, worker loss, poisoned resources, revoked scopes, stale approval, malformed messages, cancellation, exhausted budgets, and model-server overload. Assert terminal state, absence of unauthorized or duplicate effects, preserved receipts, and successful resume where promised. Then perform a canary, observe traces and service indicators, revoke a capability, roll back model and workflow versions, and publish the system card plus incident runbook.


In [ ]:
faults={"crash_before_effect":"resume","crash_after_effect":"reuse_receipt","revoked_scope":"deny","stale_approval":"reapprove","duplicate_delivery":"deduplicate","overload":"shed"}; print(*faults.items(),sep=" | ")


## 51.7 Implement the durable transition core first

Before connecting an LLM, implement a deterministic reducer from `(state,event)` to a validated next state. Events carry workflow ID, sequence, timestamp, actor, payload digest, and optional effect receipt. Reject illegal transitions and duplicate event IDs. Persist an intent before an external effect, commit its receipt afterward, and make resume query the receipt by idempotency key. Once crash and replay tests pass, place a model only in the bounded proposal transition. This separates nondeterministic planning from durable orchestration and gives single- and multi-agent variants the same trustworthy substrate for a fair capstone comparison.


In [ ]:
TRANSITIONS={("ready","plan"):"planned",("planned","start"):"running",("running","request_approval"):"waiting_approval",("waiting_approval","approve"):"running",("running","finish"):"done"}
def reduce_state(state,event):
 key=(state,event);
 if key not in TRANSITIONS: raise ValueError(f"illegal transition: {key}")
 return TRANSITIONS[key]
state="ready"
for event in ["plan","start","request_approval","approve","finish"]: state=reduce_state(state,event)
print("terminal",state); assert state=="done"


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [ReAct](https://arxiv.org/abs/2210.03629)
- [Model Context Protocol](https://modelcontextprotocol.io/specification/latest)
- [NIST AI RMF](https://www.nist.gov/itl/ai-risk-management-framework)


## Exercises

    1. Implement crash and resume tests.
2. Run matched-budget architecture ablations.
3. Publish a system card and incident drill.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
